# Práctica: Clasificación de imágenes “Gato vs No Gato”

Esta práctica está basada en un ejemplo extraído del curso de Andrew NG [Neural Networks and Deep Learning](https://www.coursera.org/learn/neural-networks-deep-learning/home/module/1) de Coursera. 

En esta práctica vamos a construir desde cero una **red neuronal completamente conectada** (*fully connected neural network*) para resolver un problema clásico de visión por computador: determinar si una imagen contiene un **gato** o **no**.

Utilizaremos el dataset [**Cat vs Non-Cat**](https://www.kaggle.com/datasets/muhammeddalkran/catvnoncat ) disponible en Kaggle.
 

Este conjunto de datos contiene imágenes en color, ya preprocesadas y divididas en entrenamiento y prueba. El objetivo será entrenar un modelo de clasificación binaria que, dada una imagen de entrada, sea capaz de predecir si contiene un gato (etiqueta 1) o no (etiqueta 0).

Para ello implementaremos manualmente todos los componentes de una red neuronal:
- Inicialización de pesos y sesgos  
- Propagación hacia delante (*forward propagation*)  
- Función de coste (entropía cruzada binaria)  
- Retropropagación (*backpropagation*)  
- Actualización de parámetros mediante descenso de gradiente  
- Evaluación del modelo y cálculo de accuracy  

Todo el proceso se realizará sin usar frameworks de alto nivel (como TensorFlow o PyTorch), lo que permitirá comprender cada detalle matemático y computacional del funcionamiento de una red neuronal.

En la siguiente celda inicializamos las librerías necesarias y cargamos un módulo auxiliar (`nn_auxiliar.py`) donde se encuentran las funciones que implementan la red.


In [ ]:
import time
import numpy as np
import h5py
import matplotlib.pyplot as plt
import scipy
from PIL import Image
from scipy import ndimage
from nn_auxiliar import *

%matplotlib inline
plt.rcParams['figure.figsize'] = (5.0, 4.0) # set default size of plots
plt.rcParams['image.interpolation'] = 'nearest'
plt.rcParams['image.cmap'] = 'gray'

%load_ext autoreload
%autoreload 2

np.random.seed(1)

## Cargamos los datos de entrada. 

En esta sección cargamos los archivos **HDF5** proporcionados por el dataset *Cat vs Non-Cat*.  
Estos archivos contienen:

- Las imágenes de entrenamiento (`train_set_x`)  
- Las etiquetas correspondientes (`train_set_y`)  
- Las imágenes de test (`test_set_x`)  
- Sus etiquetas (`test_set_y`)  
- La lista de clases (`list_classes`), donde:
  - `0` → “no gato”
  - `1` → “gato”

Cada conjunto de imágenes se carga como un array NumPy con dimensiones:

$$
(m,\; n_H,\; n_W,\; 3)
$$

donde:
- $ m $ es el número de ejemplos,  
- $ n_H \times n_W $ son las dimensiones de la imagen,  
- `3` indica que son imágenes RGB.

Las etiquetas originales vienen como un vector de dimensión `(m,)`, donde cada posición contiene `0` o `1`. Para trabajar con la notación vectorizada usada en la red neuronal, reorganizamos estas etiquetas en matrices de dimensión:

$$
(1,\; m)
$$

de modo que cada columna corresponde a una muestra, tal y como se define en nuestra implementación.

En la celda siguiente se realiza toda esta carga y reorganización del dataset.


In [ ]:
train_dataset = h5py.File('.//Datos//train_catvnoncat.h5', "r")
train_x = np.array(train_dataset["train_set_x"][:]) # Características de entrada conjunto de entrenamiento
train_y_orig = np.array(train_dataset["train_set_y"][:]) # Etiquetas (1/0) para los datos de entrenamiento 

test_dataset = h5py.File('.//Datos//test_catvnoncat.h5', "r")
test_x = np.array(test_dataset["test_set_x"][:]) # Características de entrada conjunto de test
test_y_orig = np.array(test_dataset["test_set_y"][:]) # Etiquetas (1/0) para los datos de test

classes = np.array(test_dataset["list_classes"][:]) # Lista de etiquetas
    
# Reorganizamos el conjunto de datos para que las etiquetas de las muestras se situen por columnas
train_y = train_y_orig.reshape((1, train_y_orig.shape[0]))
test_y = test_y_orig.reshape((1, test_y_orig.shape[0]))

### Exploración inicial del conjunto de datos

Antes de entrenar la red neuronal, es útil inspeccionar las dimensiones del dataset para comprender su estructura y verificar que todo se ha cargado correctamente.

Primero seleccionaremos una muestra al azar y la mostraremos junto con su etiqueta.

Además, mostramos las formas de los arrays `train_x`, `train_y`, `test_x` y `test_y` para confirmar que:

- `m_train` es el número de imágenes en el conjunto de entrenamiento.  
- `m_test` es el número de imágenes en el conjunto de test.  
- `num_px` corresponde a la altura y anchura de cada imagen, ya que todas son cuadradas de tamaño $(\,\text{num\_px} \times \text{num\_px}\,)$.  
- Las imágenes tienen 3 canales (RGB), por lo que cada una tiene dimensiones $(\text{num\_px}, \text{num\_px}, 3)$.






In [ ]:
# Ejemplo
index = np.random.randint(1,200)

plt.imshow(train_x[index])
label = train_y[0, index]

if label == 1:
    print("Es un gato! (Etiqueta = 1)")
else:
    print("No es un gato! (Etiqueta = 0)")

In [ ]:
# Analisis de las dimensiones
m_train = train_x.shape[0]
num_px = train_x.shape[1]
m_test = test_x.shape[0]

print ("Number of training examples: " + str(m_train))
print ("Number of testing examples: " + str(m_test))
print ("Each image is of size: (" + str(num_px) + ", " + str(num_px) + ", 3)")
print ("train_x shape: " + str(train_x.shape))
print ("train_y shape: " + str(train_y.shape))
print ("test_x shape: " + str(test_x.shape))
print ("test_y shape: " + str(test_y.shape))

### Reorganización y normalización de las imágenes

Las imágenes originales tienen forma:

$$
(m,\; \text{num\_px},\; \text{num\_px},\; 3)
$$

Es decir, cada imagen es un tensor 3D (alto, ancho, canales).  
Sin embargo, nuestra red neuronal totalmente conectada requiere que **cada imagen se represente como un vector columna**, donde cada fila es un píxel y cada columna corresponde a un ejemplo.



### 1. Reorganización (flattening)

Para transformar cada imagen RGB a un único vector, se utiliza:

$$
( \text{num\_px},\; \text{num\_px},\; 3 ) \quad \longrightarrow \quad (  \text{num\_px}^2 \cdot 3 ,\; 1)
$$

Esto se implementa con:

```python
train_x_flatten = train_x.reshape(train_x.shape[0], -1).T
test_x_flatten  = test_x.reshape(test_x.shape[0], -1).T
```

  - `reshape(train_x.shape[0], -1)` convierte cada imagen en un vector fila.

  - `.T` transpone, de manera que la forma final es:

Los datos reescalados pasan a tener dimensión:

$$
( d,\; m )
$$

donde:

$$
d = 3\,\text{num\_px}^2
$$

es el número total de características (píxeles) por imagen.



y cada columna representando una imagen.

### 2. Normalización de los datos

Los valores de píxel en las imágenes RGB están en el rango [0,255]. Para que la red neuronal entrene correctamente, es importante escalar estos valores a [0,1].
Esta normalización:

  - acelera el entrenamiento,

  - evita desbordamientos numéricos,

  - y mejora la estabilidad del descenso de gradiente.

In [ ]:
# Poner en el formato adecuado de (caracteristicas , muestras) 
train_x_flatten = train_x.reshape(train_x.shape[0], -1).T   # El "-1" determina que esas dimensiones se agruparan en 1
test_x_flatten = test_x.reshape(test_x.shape[0], -1).T

# Normalizar entre 0 y 1.
train_x = train_x_flatten/255.
test_x = test_x_flatten/255.

print ("train_x's shape: " + str(train_x.shape))
print ("test_x's shape: " + str(test_x.shape))

<center><img src="Imagenes/imagen_1.png" style="width:550px;height:400px;"></center>
<caption><center><font color='blue'><b>Figure 1</b>: Conversión a vector de una imagen.</font></center></caption>

Cada imagen tiene un tamaño final de 12288 "características" $12288 = 64 \times 64 \times 3$

### Diseño de la arquitectura de la red neuronal

Ahora toca definir la arquitectura completa de la red neuronal, eligiendo:

- El **número de capas ocultas**
- El **número de neuronas** en cada capa
- Las **funciones de activación** empleadas en cada una de ellas

Sin embargo, algunas condiciones están fijadas por la naturaleza del problema y del dataset:


  -  1. Dimensión de la capa de entrada `layers`. 

Las imágenes del dataset se han reorganizado (“flatten”) en vectores columna de dimensión $12288 \;=\; 64 \times 64 \times 3$
Por tanto, **la primera capa de la red debe tener exactamente 12288 neuronas**, una por cada píxel de la imagen transformada.

  -  2. Dimensión de la capa de salida.

El problema es una **clasificación binaria**:

- 1 → *gato*  
- 0 → *no gato*

Por tanto, la red debe producir **una única salida escalar**, $ \hat{y} \in [0,1]$, que representará la probabilidad de que la imagen contenga un gato. Esto implica que:

- La última capa debe tener **1 sola neurona**  
- Su función de activación debe ser **sigmoide**

Por tanto, la arquitectura siempre termina así:


   - 3. Capas ocultas y funciones de activación

Entre la capa de entrada y la de salida, puedes definir cualquier estructura:

$$
[\, 12288,\; n_1,\; n_2,\; \dots,\; n_{L-1},\; 1 \,]
$$

donde los $n_i$ son el número de neuronas de cada capa oculta.

Las activaciones `activations` y sus derivadas `activations_prime` también son elegibles para cada capa, aunque típicamente se recomienda:

- **ReLU** para capas ocultas:  `relu` y su derivada `relu_prime`
- **Sigmoid** en la última capa: `sigmoid` y su derivada `sigmoid_prime`



In [ ]:
activations       = [relu, relu, relu, sigmoid]
activations_prime = [relu_prime, relu_prime, relu_prime, sigmoid_prime]
layers = [12288, 20, 7, 5, 1]


<center><img src="Imagenes/imagen_2.png" style="width:750px;height:500px;"></center>
<caption><center><font color='blue'><b>Figure 2</b>: Red neuronal de L capas <br> </font></center></caption>

### Entrenamiento

Completa la siguiente función con las llamadas correctas a:

`model_forward(...)`

`model_backward(...)`

`update_params(...)`

`compute_cost(...)`

El modelo se entrenará con descenso de gradiente estándar, utilizando las dimensiones especificadas en `layers` y las funciones que hayas elegido.
En este proceso:

- El *forward propagation* calcula la predicción  $A^{[L]} $
- La función de coste evalúa el error entre $ A^{[L]} $ y $ Y $
- El *backpropagation* calcula los gradientes de todas las capas
- Finalmente, los parámetros se actualizan en la dirección que reduce el coste


In [ ]:
def train_network(X, Y, layers, activations, activations_prime,
                  learning_rate=0.01, epochs=1000, seed=None, print_cost=False):

    perdida = []
    params = initialize_params(layers, seed=seed)

    for epoch in range(epochs):
        # ********** Tu codigo aquí ***************************
        #  Forward
        #  Backward
        #  Descenso de gradiente
        #  Calculo coste
        # ********** Tu codigo aquí ***************************

        if print_cost and (epoch % 100 == 0 or epoch == epochs - 1):
            print(f"Cost after iteration {epoch}: {coste}")

    return params, perdida


### Entrenamiento del modelo

Una vez definida la arquitectura de la red y preparadas todas las funciones auxiliares, ya podemos entrenar nuestro modelo utilizando el conjunto de entrenamiento.


En la siguiente celda se ejecuta:

```python
parameters, costs = train_network(train_x, train_y,
                                  layers,
                                  activations,
                                  activations_prime,
                                  learning_rate = 0.0075,
                                  epochs        = 3500,
                                  print_cost    = True)
```
Durante el entrenamiento se imprimirá el valor del coste cada cierto número de iteraciones (si `print_cost=True`), lo que permite comprobar si la red está aprendiendo correctamente y si el coste disminuye de forma estable.

Al finalizar:

  - `parameters`  contendrá los pesos y sesgos aprendidos por la red.

  - `costs` almacenará la evolución del coste durante el entrenamiento, útil para visualizar el progreso y detectar posibles problemas como saturación o divergencia.


In [ ]:
parameters, costs = train_network(train_x, train_y, layers, activations, activations_prime, learning_rate=0.0075, epochs = 3500, print_cost = True)

### Evaluación del modelo en el conjunto de entrenamiento

Tras finalizar el entrenamiento de la red neuronal, es fundamental evaluar su rendimiento sobre los datos utilizados para el aprendizaje. Esto permite verificar si la red ha logrado ajustar correctamente los parámetros y si ha aprendido a distinguir entre imágenes de *gato* y *no gato*.

Para ello utilizamos la función `predict`, que:

1. Realiza un *forward propagation* con los parámetros ya entrenados.  
2. Genera predicciones binarias aplicando un umbral (por defecto, 0.5).  
3. Compara estas predicciones con las etiquetas reales.  
4. Calcula la **accuracy** (proporción de aciertos).




In [ ]:
train_preds, train_acc = predict(train_x, train_y, parameters, activations)
print("Train accuracy:", train_acc)

### Evaluación del modelo en el conjunto de test

Después de comprobar el rendimiento del modelo en los datos de entrenamiento, el siguiente paso es evaluar su capacidad para **generalizar** a datos que nunca ha visto. Para ello utilizamos el conjunto de test.

La función `predict` se aplica ahora sobre `test_x` y `test_y`:

In [ ]:
test_preds, test_acc = predict(test_x, test_y, parameters, activations)
print("Test accuracy:", test_acc)


### Visualización de imágenes mal clasificadas

Para comprender mejor los errores del modelo y analizar en qué tipos de imágenes falla, es útil inspeccionar visualmente aquellos ejemplos en los que la predicción no coincide con la etiqueta real.

La función `print_mislabeled_images` muestra precisamente estas imágenes, organizadas en filas de hasta 6 imágenes por fila. Para cada imagen se indican:

- La **predicción** del modelo (gato / no gato)  
- La **etiqueta real** correspondiente  

Esto permite detectar patrones en los errores, por ejemplo:

- Imágenes oscuras o borrosas  
- Posiciones poco habituales del gato  
- Objetos o texturas parecidas a un gato  
- Confusiones sistemáticas en ciertos casos  

La siguiente celda ejecuta esta visualización sobre el conjunto de test:


In [ ]:
def print_mislabeled_images(X, y, p):
    """
    Muestra las imágenes mal clasificadas organizadas en filas de máximo 6 imágenes.

    X -- conjunto de imágenes (aplanadas)
    y -- etiquetas reales
    p -- predicciones del modelo
    """

    # Indices donde predicción y etiqueta real son distintas
    mislabeled_indices = np.where(p != y)
    mislabeled_indices = mislabeled_indices[1]   # extraer vector de índices

    num_images = len(mislabeled_indices)
    if num_images == 0:
        print("No hay imágenes mal clasificadas. ¡Excelente!")
        return

    max_cols = 6
    rows = int(np.ceil(num_images / max_cols))

    plt.figure(figsize=(max_cols * 3, rows * 3))

    for i, index in enumerate(mislabeled_indices):
        plt.subplot(rows, max_cols, i + 1)

        img = X[:, index].reshape(64, 64, 3)
        plt.imshow(img, interpolation='nearest')
        plt.axis('off')

        true_label = y[0, index]
        pred_label = p[0, index]

        if pred_label == 1:
            pred_text = "Predicción: Gato"
        else:
            pred_text = "Predicción: No gato"

        if true_label == 1:
            true_text = "Real: Gato"
        else:
            true_text = "Real: No gato"

        plt.title(pred_text + "\n" + true_text)

    plt.show()
    
print_mislabeled_images(test_x, test_y, test_preds)


### Probando el modelo con una imagen propia

Una vez entrenada la red neuronal, podemos utilizarla para clasificar imágenes nuevas que **no pertenecen al dataset original**.  
Para ello puedes:

1. Seleccionar una imagen (preferiblemente de un gato o un objeto que pueda confundirse con un gato).  
2. Guardarla dentro de la carpeta **`Imagenes/`** del proyecto.  
3. Indicar el nombre del archivo en la celda de código correspondiente.  
4. Especificar la etiqueta real de la imagen:  
   - `1` → si la imagen contiene un **gato**  
   - `0` → si **no contiene un gato**

La red realizará los mismos pasos de preprocesado que con las imágenes del dataset (redimensionado, aplanado y normalización) y devolverá una predicción indicando si, según el modelo, “es un gato” o “no es un gato”.


In [ ]:
my_image = "minogato.jpg"        # Cambiar por tu archivo
my_label_y = np.array([[0]])     # 1 -> gato, 0 -> no gato

fname = "Imagenes/" + my_image
plt.figure(figsize=(3, 3))
image = np.array(Image.open(fname).resize((num_px, num_px)))
plt.imshow(image)

# Normalizar y reordenar
image = image / 255.
image = image.reshape((1, num_px * num_px * 3)).T

# Predicción (nota: predict devuelve predicción y accuracy)
my_pred, _ = predict(image, my_label_y, parameters, activations)

if np.squeeze(my_pred) == 1:
    pred_text = "es un gato"
else:
    pred_text = "no es un gato"

print("Tu modelo predice que la imagen " + pred_text)
